# 06 — NYC Taxi Zones Bronze to Silver

## 1. Read Taxi Zone Lookup

Read the NYC Taxi Zone Lookup CSV from the Bronze layer.

In [ ]:
df_zones = (
    spark.read
    .option("header", True)
    .csv("Files/bronze/taxi_zones/taxi_zone_lookup.csv")
)

df_zones.show(5)
df_zones.printSchema()

## 2. Transform Taxi Zone Data

Convert LocationID to integer so it can be matched with taxi pickup and dropoff location IDs.

In [ ]:
df_zones_silver = df_zones.withColumn(
    "LocationID",
    df_zones.LocationID.cast("integer")
)

In [ ]:
df_zones_silver.printSchema()
df_zones_silver.show(5)

## 3. Data Quality Checks

Validate the total number of taxi zones, missing LocationIDs, and duplicate LocationIDs.

In [ ]:
total_zones = df_zones_silver.count()

print("Total Zones: ", total_zones)

In [ ]:
null_location_count = df_zones_silver.filter(df_zones_silver.LocationID.isNull()).count()

print("Null count location: ", null_location_count)

In [ ]:
unique_zones = df_zones_silver.dropDuplicates(["LocationID"]).count()

print("Unique Zones: ", unique_zones)

In [ ]:
duplicate_location_count = total_zones - unique_zones

print("Duplicate LocationID:", duplicate_location_count)

## 4. Validate Taxi Zone Mapping

Check whether taxi pickup and dropoff location IDs exist in the Taxi Zone Lookup table.

In [ ]:
df_taxi = spark.table("silver_taxi_trips")

pu_ids = df_taxi.select("PULocationID").distinct()

In [ ]:
missing_pu = pu_ids.join(
    df_zones_silver,
    pu_ids.PULocationID == df_zones_silver.LocationID,
    "left_anti"
)

missing_pu.show()

In [ ]:
do_ids = df_taxi.select("DOLocationID").distinct()
missing_do = do_ids.join(
    df_zones_silver,
    do_ids.DOLocationID == df_zones_silver.LocationID,
    "left_anti"
)

missing_do.show()

## 5. Write Silver Delta Table

Save the transformed NYC Taxi Zone Lookup as a Delta table in the Lakehouse.

In [ ]:
(
    df_zones_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver_taxi_zones")
)

## 6. Validate Silver Output

Verify the saved table, row count, column count, and invalid zone names.

In [ ]:
df_check = spark.table("silver_taxi_zones")

print("Total Rows:", df_check.count())
print("Total Columns:", len(df_check.columns))

In [ ]:
from pyspark.sql.functions import col, trim

invalid_zone_count = df_check.filter(
    col("Zone").isNull()
    |
    (trim(col("Zone")) == "")
).count()

print("Invalid Zone Count: ", invalid_zone_count)